# ML4NLP1
## Starting Point for Exercise 1, part II

This notebook is supposed to serve as a starting point and/or inspiration when starting exercise 1, part II.

One of the goals of this exercise is o make you acquainted with **skorch**. You will probably need to consult the [documentation](https://skorch.readthedocs.io/en/stable/).

# Installing skorch and loading libraries

In [1]:
import subprocess

# Installation on Google Colab
try:
    import google.colab
    subprocess.run(['python', '-m', 'pip', 'install', 'skorch'])
except ImportError:
    pass

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from skorch import NeuralNetClassifier

import pandas as pd
import numpy as np

# Set seed for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

## Training a classifier and making predictions

In [ ]:
# Download dataset
!gdown 1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs # x_train
!gdown 1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6 # x_test
!gdown 1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl # y_train
!gdown 1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X # y_test

Downloading...
From: https://drive.google.com/uc?id=1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs
To: /content/x_train.txt
100% 64.1M/64.1M [00:01<00:00, 49.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6
To: /content/x_test.txt
100% 65.2M/65.2M [00:00<00:00, 187MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl
To: /content/y_train.txt
100% 480k/480k [00:00<00:00, 92.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X
To: /content/y_test.txt
100% 480k/480k [00:00<00:00, 94.2MB/s]


In [3]:
with open(f'x_train.txt') as f:
    x_train = f.read().splitlines()
with open(f'y_train.txt') as f:
    y_train = f.read().splitlines()
with open(f'x_test.txt') as f:
    x_test = f.read().splitlines()
with open(f'y_test.txt') as f:
    y_test = f.read().splitlines()

In [4]:
# Combine x_train and y_train into one dataframe
train_df = pd.DataFrame({'text': x_train, 'label': y_train})
# Write train_df to csv with tab as separator
train_df.to_csv('train_df.csv', index=False, sep='\t')
# Comibne x_test and y_test into one dataframe
test_df = pd.DataFrame({'text': x_test, 'label': y_test})
# Inspect the first 5 items in the train split
train_df.head()

,text,label
0,Klement Gottwaldi surnukeha palsameeriti ning ...,est
1,"Sebes, Joseph; Pereira Thomas (1961) (på eng)....",swe
2,भारतीय स्वातन्त्र्य आन्दोलन राष्ट्रीय एवम क्षे...,mai
3,"Après lo cort periòde d'establiment a Basilèa,...",oci
4,ถนนเจริญกรุง (อักษรโรมัน: Thanon Charoen Krung...,tha


### Data preparation

Prepare your dataset for this experiment using the same method as you did in part 1.

Get a subset of the train/test data that includes 20 languages. Include English, German, Dutch, Danish, Swedish, Norwegian, and Japanese, plus 13 additional languages of your choice based on the items in the list of labels.

Don't forget to encode your labels using the adjusted code snippet from part 1!


In [5]:
# TODO: Create your train/test subsets of languages
# Note, make sure these are the same as what you used in Part 1!

# Prepare train/test subsets with the same 20 languages as Part 1
from sklearn.model_selection import train_test_split

langs20 = [
    'eng', 'deu', 'nld', 'dan', 'swe', 'nob', 'jpn', 'est', 'mai', 'oci',
    'tha', 'lim', 'guj', 'zea', 'krc', 'hat', 'tam', 'vie', 'pan', 'ckb'
]

# After filtering the data at both ends, divide them uniformly to ensure consistency with Part 1
tr_sel = train_df.loc[train_df['label'].isin(langs20)].copy()
ts_sel = test_df.loc[test_df['label'].isin(langs20)].copy()

all_data = pd.concat([tr_sel, ts_sel], ignore_index=True)
train_df_adjusted, test_df_adjusted = train_test_split(
    all_data, test_size=0.2, random_state=42
)

X_train = train_df_adjusted['text'].to_numpy()
X_test = test_df_adjusted['text'].to_numpy()
y_train = train_df_adjusted['label'].to_numpy()
y_test = test_df_adjusted['label'].to_numpy()

print(np.unique(y_train))

['ckb' 'dan' 'deu' 'eng' 'est' 'guj' 'hat' 'jpn' 'krc' 'lim' 'mai' 'nld'
 'nob' 'oci' 'pan' 'swe' 'tam' 'tha' 'vie' 'zea']


In [6]:
# Encode labels (fit on training labels only, then transform test labels)
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train = label_encoder.transform(y_train)
y_test = label_encoder.transform(y_test)

print(label_encoder.classes_)
print(y_train)
print(y_test)

['ckb' 'dan' 'deu' 'eng' 'est' 'guj' 'hat' 'jpn' 'krc' 'lim' 'mai' 'nld'
 'nob' 'oci' 'pan' 'swe' 'tam' 'tha' 'vie' 'zea']
[ 0  6 16 ... 16 10 18]
[ 7 15  8 ...  1 15 11]


### Feature Extraction

In [7]:
# First, we extract some simple features as input for the neural network
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 2), max_features=100, binary=True)
X = vectorizer.fit_transform(X_train)

In [8]:
# We need to change the datatype to make it play nice with pytorch
X = X.astype(np.float32)
y = y_train.astype(np.int64)

In [9]:
print(X.shape)
print(y.shape)
print(len(label_encoder.classes_))
print(np.unique(y_train))
print(X.shape[1])
print(pd.Series(y_train).value_counts())

(16000, 100)
(16000,)
20
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
100
17    820
15    815
19    815
10    808
11    806
1     806
7     804
4     803
6     801
5     800
16    800
14    800
0     796
9     796
18    793
13    792
8     789
12    787
2     786
3     783
Name: count, dtype: int64


In the following, we define a vanilla neural network with two hidden layers. The output layer should have as many outputs as there are classes. In addition, it should have a nonlinearity function.

In [10]:
# In the following, you can find a small (almost) working example of a neural network.
# Unfortunately, again, the cat messed up some of the code. Please fix the code such that it is executable.
# (Hint: the input and output sizes look a bit weird...)

# Input Dimension for our dataset
input_size = X.shape[1]

class ClassifierModule(nn.Module):
    def __init__(
        self,
        input_size,
        num_classes,
        num_units=200,
        nonlin=F.relu,
    ):
        super(ClassifierModule, self).__init__()
        self.num_units = num_units
        self.nonlin = nonlin

        self.dense0 = nn.Linear(input_size, num_units)
        self.nonlin = nonlin
        self.dense1 = nn.Linear(num_units, num_units)
        self.output = nn.Linear(num_units, 20)

    def forward(self, X, **kwargs):
        X = self.nonlin(self.dense0(X))
        X = F.relu(self.dense1(X))
        X = self.output(X)
        return X.squeeze(dim=1)


In [11]:
# Initalise the neural net classifier.
net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=F.relu,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cpu',  # comment this to train with CPU
)

In [12]:
# Train the classifier
net.fit(X, y)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        2.6668       0.3556        2.3281  1.7703
      2        1.9095       0.4381        1.6078  1.6414
      3        1.4404       0.5356        1.3231  1.8181
      4        1.2405       0.6275        1.1713  2.3862
      5        1.1085       0.6375        1.0618  2.2180
      6        1.0094       0.6475        0.9919  2.2028
      7        0.9454       0.6706        0.9542  2.0378
      8        0.9082       0.6737        0.9348  1.8901
      9        0.8853       0.6822        0.9236  1.8022
     10        0.8693       0.6834        0.9163  1.8101
     11        0.8570       0.6853        0.9106  1.8156
     12        0.8468       0.6891        0.9067  2.0555
     13        0.8381       0.6913        0.9033  1.8157
     14        0.8303       0.6906        0.9005  1.8045
     15        0.8233       0.6903        0.8981  1.8046
     16        0.8169       0.6

<class 'skorch.classifier.NeuralNetClassifier'>[initialized](
  module_=ClassifierModule(
    (dense0): Linear(in_features=100, out_features=200, bias=True)
    (dense1): Linear(in_features=200, out_features=200, bias=True)
    (output): Linear(in_features=200, out_features=20, bias=True)
  ),
)

Note, you can also use `GridSearchCV` with `skorch`, but be aware that training a neural network takes much more time.

Play around with 5 different sets of hyperparameters. For example, consider some of the following:

- layer sizes
- activation functions
- regularizers
- early stopping
- vectorizer parameters

Report your best hyperparameter combination. \\
📝❓ What is the effect of your modifcations on validation performance? Discuss potential reasons.

☝ Note, during model development, if you run into the infamous CUDA out-of-memory (OOM) error, try clearing the GPU memory either with `torch.cuda.empty_cache()` or restarting the runtime.

In [13]:
# First, we extract some simple features as input for the neural network
from sklearn.feature_extraction.text import TfidfVectorizer
from skorch.callbacks import EarlyStopping

tfidf_vec = TfidfVectorizer(max_features=100)
X_tfidf = tfidf_vec.fit_transform(X_train).astype(np.float32)

# Initialise the neural net classifier (same setup, different variable names)
early_stopping = EarlyStopping(patience=3, threshold=0.1)

net_2 = NeuralNetClassifier(
    ClassifierModule(
        input_size=X_tfidf.shape[1],
        num_units=100,
        num_classes=len(label_encoder.classes_),
        nonlin=F.leaky_relu,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.2,
    device='cpu',
    callbacks=[early_stopping],
)

net_2.fit(X_tfidf, y)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        2.9819       0.2537        2.9606  1.6796
      2        2.9044       0.5212        2.7890  1.5761
      3        2.3383       0.7297        1.6386  1.6008
      4        1.0694       0.8159        0.7171  2.1436
      5        0.6080       0.8438        0.5432  2.0149
      6        0.4994       0.8531        0.4826  2.3031
      7        0.4500       0.8606        0.4502  2.2928
      8        0.4209       0.8662        0.4308  2.2361
      9        0.4021       0.8684        0.4179  1.8977
     10        0.3886       0.8709        0.4081  1.7918
Stopping since valid_loss has not improved in the last 3 epochs.


<class 'skorch.classifier.NeuralNetClassifier'>[initialized](
  module_=ClassifierModule(
    (dense0): Linear(in_features=100, out_features=100, bias=True)
    (dense1): Linear(in_features=100, out_features=100, bias=True)
    (output): Linear(in_features=100, out_features=20, bias=True)
  ),
)


---
## Lab Report

📝❓ Write your lab report here addressing all questions in the notebook

> Report your best hyperparameter combination.
> 📝❓ What is the effect of your modifcations on validation performance? Discuss potential reasons.

Feature reduction yields computational efficiency at the cost of representational capacity; paradoxically, this may improve generalization by suppressing noise and limiting overfitting.
Early stopping halts training once validation metrics no longer improve, thereby reducing both overfitting risk and compute, though given the minimal drift in the initial model it is unlikely the principal source of improvement.
Employing TF–IDF alters term importance via inverse document frequency, potentially enhancing discriminative power.

## AI Content Declaration

Portions of the content in this Jupyter notebook were generated with the assistance of an artificial intelligence model. Specifically, AI was used to assist in:

- debugging errors
- Providing explanations and clarifications for complex concepts.

All outputs and interpretations were reviewed and verified for accuracy, but the user should take responsibility for the correctness and appropriateness of the code and results.

The AI model used for assistance is OpenAI's ChatGPT 4o model.
